**Row Count validation**

In [0]:
SELECT COUNT(*) FROM retail_lakehouse.bronze.sales;
SELECT COUNT(*) FROM retail_lakehouse.silver.sales;
SELECT COUNT(*) FROM retail_lakehouse.gold.fact_sales;

**Null validation**

In [0]:
SELECT *
FROM retail_lakehouse.silver.sales
WHERE TransactionID IS NULL;

**Duplicate validation**

In [0]:
SELECT TransactionID, COUNT(*)
FROM retail_lakehouse.silver.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

**Join validation**

In [0]:
SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;

**Column level mapping**

In [0]:
SELECT
    b.CustomerID AS bronze_customer_id,
    s.CustomerID AS silver_customer_id,
    b.CustomerName AS bronze_customer_name,
    s.CustomerName AS silver_customer_name
FROM retail_lakehouse.bronze.customers b
JOIN retail_lakehouse.silver.customers s
ON b.CustomerID = s.CustomerID
LIMIT 10;

**Data type validation and derived data validation**

In [0]:
DESCRIBE retail_lakehouse.gold.fact_sales;
DESCRIBE retail_lakehouse.silver.sales;

**Data transformation testing**

In [0]:
SELECT
    CustomerName
FROM retail_lakehouse.silver.customers
WHERE CustomerName != INITCAP(CustomerName);


In [0]:
SELECT Email
FROM retail_lakehouse.silver.customers
WHERE Email != LOWER(Email);

In [0]:
SELECT TxnDate
FROM retail_lakehouse.silver.sales
LIMIT 10;

In [0]:
SELECT
    s.Quantity,
    p.UnitPrice,
    f.Amount
FROM retail_lakehouse.gold.fact_sales f
JOIN retail_lakehouse.silver.sales s
ON f.TransactionID = s.TransactionID
JOIN retail_lakehouse.silver.products p
ON s.ProductID = p.ProductID
LIMIT 10;

**Referential integrity**

In [0]:
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_customer c
ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

In [0]:
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_product p
ON s.ProductID = p.ProductID
WHERE p.ProductID IS NULL;

In [0]:
SELECT *
FROM retail_lakehouse.silver.sales s
LEFT JOIN retail_lakehouse.gold.dim_store st
ON s.StoreID = st.StoreID
WHERE st.StoreID IS NULL;

**Data quality check**

In [0]:
-- -----------------------------------------
-- Duplicate Validation
-- -----------------------------------------

SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.silver.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;
-- Expected Result:
-- 0 rows

-- -----------------------------------------
-- Null Validation
-- -----------------------------------------

SELECT *
FROM retail_lakehouse.silver.sales
WHERE TransactionID IS NULL;

-- Expected Result:
-- 0 rows

-- -----------------------------------------
-- Invalid Data Validation
-- -----------------------------------------

SELECT *
FROM retail_lakehouse.silver.sales
WHERE Quantity <= 0;

-- Expected Result:
-- 0 rows

SELECT *
FROM retail_lakehouse.silver.products
WHERE UnitPrice <= 0;

-- Expected Result:
-- 0 rows


**SCD type 2 validation**

In [0]:
-- =========================================
-- SECTION 5 — SCD TYPE 2 VALIDATION
-- =========================================

-- -----------------------------------------
-- Active vs Inactive Validation
-- -----------------------------------------

SELECT
    CustomerID,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate;

-- -----------------------------------------
-- Only One Active Record Validation
-- -----------------------------------------

SELECT
    CustomerID,
    COUNT(*)
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

-- Expected Result:
-- 0 rows

**CDC Validation**

In [0]:
-- =========================================
-- SECTION 6 — CDC VALIDATION
-- =========================================
-- describe history retail_lakehouse.silver.sales;
-- SELECT
--     _change_type,
--     _commit_version,
--     _commit_timestamp,
--     TransactionID
-- FROM table_changes(
--     'retail_lakehouse.silver.sales',
--     17
-- )
-- ORDER BY _commit_version DESC;


**Full load testing**

In [0]:
-- =========================================
-- SECTION 7 — FULL LOAD vs INCREMENTAL LOAD
-- =========================================

-- -----------------------------------------
-- Full Load Validation
-- -----------------------------------------

SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;

-- -----------------------------------------
-- Incremental Load Validation
-- -----------------------------------------

SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC LIMIT 10;

-- Verify:
-- Only incremental rows added
-- No full reload duplicates

In [0]:
SELECT
    s.TransactionID,
    s.Quantity,
    p.UnitPrice,
    (s.Quantity * p.UnitPrice) AS ExpectedAmount,
    f.Amount AS ActualAmount
FROM retail_lakehouse.gold.fact_sales f
JOIN retail_lakehouse.silver.sales s
ON f.TransactionID = s.TransactionID
JOIN retail_lakehouse.silver.products p
ON s.ProductID = p.ProductID
ORDER BY s.TransactionID DESC
LIMIT 10;

In [0]:
-- =========================================
-- SECTION 2:
-- INCREMENTAL LOAD VALIDATION
-- =========================================

-- INSERT NEW INCREMENTAL RECORD

INSERT INTO retail_lakehouse.silver.sales
VALUES
(
    9999,
    1,
    1001,
    100,
    5,
    CURRENT_DATE()
);

-- =========================================
-- RUN:
-- 05_incremental_load.sql
-- =========================================

-- VERIFY NEW RECORD INSERTED

SELECT *
FROM retail_lakehouse.gold.fact_sales
WHERE TransactionID = 9999;

-- =========================================
-- VERIFY NO DUPLICATE INSERTS
-- =========================================

SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.gold.fact_sales
WHERE TransactionID = 9999
GROUP BY TransactionID;

-- EXPECTED:
-- COUNT = 1

-- =========================================
-- VERIFY INCREMENTAL UPDATE
-- =========================================

UPDATE retail_lakehouse.silver.sales
SET Quantity = 10
WHERE TransactionID = 9999;

-- =========================================
-- RUN:
-- 05_incremental_load.sql AGAIN
-- =========================================

SELECT
    TransactionID,
    Quantity,
    Amount
FROM retail_lakehouse.gold.fact_sales
WHERE TransactionID = 9999;

-- EXPECTED:
-- Quantity = 10
-- Amount recalculated

-- =========================================
-- SECTION 3:
-- SCD TYPE 2 VALIDATION
-- =========================================

-- UPDATE CUSTOMER ATTRIBUTE

UPDATE retail_lakehouse.silver.customers
SET City = 'Bangalore'
WHERE CustomerID = 1;

-- =========================================
-- RUN:
-- 06_scd2_processing.sql
-- =========================================

-- VERIFY HISTORY CREATED

SELECT
    CustomerID,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate;

-- EXPECTED:
-- old row inactive
-- new row active

-- =========================================
-- VERIFY ONLY ONE ACTIVE RECORD
-- =========================================

SELECT
    CustomerID,
    COUNT(*)
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

-- EXPECTED:
-- 0 rows

-- =========================================
-- VERIFY ENDDATE LOGIC
-- =========================================

SELECT
    CustomerID,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

-- EXPECTED:
-- inactive row EndDate = current date
-- active row EndDate = 9999-12-31